In [2]:
import pandas as pd
import numpy as np
import glob

In [3]:
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
orders = pd.read_csv("orders_raw.csv")
order_items = pd.read_csv("order_items_raw.csv")
payments = pd.read_csv("payments.csv")
shipping = pd.read_csv("shipping.csv")
returns = pd.read_csv("returns.csv")

## 1.Load all sheets and inspect shape/info/dtypes


In [4]:
tables = {
    "customers":customers,
    "products":products,
    "orders_raw":orders,
    "orders_items_raw":order_items,
    "payments":payments,
    "shipping":shipping,
    "returns":returns
}
for name, df in tables.items():
    print(f"\n--- {name.upper()} ---")
    print("Shape:", df.shape)
    print("Info:",df.info())
    print("Data Types:",df.dtypes)
    print("Duplicates:", df.duplicated().sum())
    print("Nulls:")
    print(df.isna().sum())



--- CUSTOMERS ---
Shape: (1200, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   customer_id          1200 non-null   str  
 1   customer_name        1200 non-null   str  
 2   signup_date          1200 non-null   str  
 3   segment              1200 non-null   str  
 4   city                 1200 non-null   str  
 5   region               1200 non-null   str  
 6   acquisition_channel  1200 non-null   str  
dtypes: str(7)
memory usage: 65.8 KB
Info: None
Data Types: customer_id            str
customer_name          str
signup_date            str
segment                str
city                   str
region                 str
acquisition_channel    str
dtype: object
Duplicates: 0
Nulls:
customer_id            0
customer_name          0
signup_date            0
segment                0
city                   0
region                 0
acq

## 2.Check nulls, duplicates and unique values

In [5]:
for name, df in tables.items():
    print(f"\n--- {name.upper()} ---")
    print(df.isna().sum())


--- CUSTOMERS ---
customer_id            0
customer_name          0
signup_date            0
segment                0
city                   0
region                 0
acquisition_channel    0
dtype: int64

--- PRODUCTS ---
product_id      0
product_name    0
category        0
subcategory     0
unit_cost       0
list_price      0
product_tier    0
dtype: int64

--- ORDERS_RAW ---
order_id          0
customer_id       0
order_date        0
sales_channel     0
payment_method    0
order_status      0
dtype: int64

--- ORDERS_ITEMS_RAW ---
order_item_id    0
order_id         0
product_id       0
quantity         0
discount_pct     0
gross_sales      0
net_sales        0
profit           0
dtype: int64

--- PAYMENTS ---
order_id            0
payment_method      0
payment_status      0
transaction_date    0
order_amount        0
dtype: int64

--- SHIPPING ---
order_id         0
shipping_mode    0
dispatch_date    0
delivery_date    0
shipping_cost    0
dtype: int64

--- RETURNS ---
order_id

In [6]:
orders["sales_channel"].unique()
orders["order_status"].unique()

products["category"].unique()
products["subcategory"].unique()
products["product_tier"].unique()

payments["payment_method"].unique()
payments["payment_status"].unique()

shipping["shipping_mode"].unique()

returns["return_reason"].unique()

<StringArray>
[  'Quality Issue',    'Changed Mind',   'Late Delivery',  'Size/Fit Issue',
      'Wrong Item', 'Damaged Product']
Length: 6, dtype: str

## 3. Clean text categories

In [7]:
orders["sales_channel"] = orders["sales_channel"].str.strip().str.title()

In [8]:
orders["order_status"] = orders["order_status"].str.strip().str.title()

## Remove the known duplicate rows

In [9]:
orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()

In [10]:
print(orders["order_id"].duplicated().sum())
print(order_items["order_item_id"].duplicated().sum())

0
0


## 4. Validate foreign keys

In [11]:
invalid_customers = orders[~orders["customer_id"].isin(customers["customer_id"])]
print(invalid_customers)

     order_id customer_id  order_date sales_channel payment_method  \
25  ORD000026   CUST99999  2025-04-15    Mobile App            UPI   

   order_status  
25    Completed  


In [12]:
invalid_orders = order_items[~order_items["order_id"].isin(orders["order_id"])]
print(invalid_orders)

Empty DataFrame
Columns: [order_item_id, order_id, product_id, quantity, discount_pct, gross_sales, net_sales, profit]
Index: []


In [13]:
invalid_products = order_items[~order_items["product_id"].isin(products["product_id"])]
print(invalid_products)

Empty DataFrame
Columns: [order_item_id, order_id, product_id, quantity, discount_pct, gross_sales, net_sales, profit]
Index: []


In [14]:
invalid_payment_orders = payments[~payments["order_id"].isin(orders["order_id"])]
print(invalid_payment_orders)

Empty DataFrame
Columns: [order_id, payment_method, payment_status, transaction_date, order_amount]
Index: []


In [15]:
invalid_shipping_orders = shipping[~shipping["order_id"].isin(orders["order_id"])]
print(invalid_shipping_orders)

Empty DataFrame
Columns: [order_id, shipping_mode, dispatch_date, delivery_date, shipping_cost]
Index: []


In [16]:
invalid_return_orders = returns[~returns["order_id"].isin(orders["order_id"])]
print(invalid_return_orders)

Empty DataFrame
Columns: [order_id, return_date, return_reason, refund_amount]
Index: []


## 5. Convert dates

In [17]:
customers["signup_date"] = pd.to_datetime(customers["signup_date"])

orders["order_date"] = pd.to_datetime(orders["order_date"])

payments["transaction_date"] = pd.to_datetime(payments["transaction_date"])

shipping["dispatch_date"] = pd.to_datetime(shipping["dispatch_date"])
shipping["delivery_date"] = pd.to_datetime(shipping["delivery_date"])

returns["return_date"] = pd.to_datetime(returns["return_date"])


In [18]:
orders.dtypes

order_id                     str
customer_id                  str
order_date        datetime64[us]
sales_channel                str
payment_method               str
order_status                 str
dtype: object

## 6.Create month/year/delivery_days

In [19]:
orders["year"] = orders["order_date"].dt.year

In [20]:
orders["month"] = orders["order_date"].dt.month

In [21]:
orders["year_month"] = orders["order_date"].dt.to_period("M")

In [22]:
shipping["delivery_days"] = (shipping["delivery_date"] - shipping["dispatch_date"]).dt.days

In [23]:
#verify
shipping[[
    "dispatch_date",
    "delivery_date",
    "delivery_days"
]].head()

,dispatch_date,delivery_date,delivery_days
0,2026-05-29,2026-06-06,8
1,2025-11-03,2025-11-09,6
2,2026-01-23,2026-01-29,6
3,2024-11-19,2024-11-23,4
4,2025-02-27,2025-02-28,1


## 7. Merge Orders + Items + Customers + Products

In [24]:
df=orders.merge(customers,on="customer_id",how="left")
df.shape
df.head()

,order_id,customer_id,order_date,sales_channel,payment_method,order_status,year,month,year_month,customer_name,signup_date,segment,city,region,acquisition_channel
0,ORD000001,CUST00559,2026-05-28,Website,UPI,Completed,2026,5,2026-05,Customer 00559,2024-01-30,Consumer,Ahmedabad,West,Email
1,ORD000002,CUST01197,2025-11-01,Website,Net Banking,Completed,2025,11,2025-11,Customer 01197,2023-05-27,Consumer,Kolkata,East,Email
2,ORD000003,CUST00400,2026-01-21,Website,Debit Card,Returned,2026,1,2026-01,Customer 00400,2025-01-17,Consumer,Kolkata,East,Organic Search
3,ORD000004,CUST01122,2024-11-16,Website,UPI,Completed,2024,11,2024-11,Customer 01122,2022-03-15,Consumer,Hyderabad,South,Referral
4,ORD000005,CUST00201,2025-02-25,Mobile App,Net Banking,Completed,2025,2,2025-02,Customer 00201,2025-03-26,Consumer,Pune,West,Email


In [25]:
df=df.merge(order_items,on="order_id",how="left")
df.shape
df.head()

,order_id,customer_id,order_date,sales_channel,payment_method,order_status,year,month,year_month,customer_name,...,city,region,acquisition_channel,order_item_id,product_id,quantity,discount_pct,gross_sales,net_sales,profit
0,ORD000001,CUST00559,2026-05-28,Website,UPI,Completed,2026,5,2026-05,Customer 00559,...,Ahmedabad,West,Email,ITEM0003414,PROD0107,1.0,0.10,14258.33,12832.50,3503.51
1,ORD000001,CUST00559,2026-05-28,Website,UPI,Completed,2026,5,2026-05,Customer 00559,...,Ahmedabad,West,Email,ITEM0004492,PROD0001,1.0,0.05,4107.99,3902.59,453.62
2,ORD000001,CUST00559,2026-05-28,Website,UPI,Completed,2026,5,2026-05,Customer 00559,...,Ahmedabad,West,Email,ITEM0005520,PROD0220,2.0,0.10,48513.80,43662.42,9232.80
3,ORD000002,CUST01197,2025-11-01,Website,Net Banking,Completed,2025,11,2025-11,Customer 01197,...,Kolkata,East,Email,ITEM0000945,PROD0079,1.0,0.15,8940.37,7599.31,2277.79
4,ORD000003,CUST00400,2026-01-21,Website,Debit Card,Returned,2026,1,2026-01,Customer 00400,...,Kolkata,East,Organic Search,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df=df.merge(products,on="product_id",how="left")
df.shape
df.head()


,order_id,customer_id,order_date,sales_channel,payment_method,order_status,year,month,year_month,customer_name,...,discount_pct,gross_sales,net_sales,profit,product_name,category,subcategory,unit_cost,list_price,product_tier
0,ORD000001,CUST00559,2026-05-28,Website,UPI,Completed,2026,5,2026-05,Customer 00559,...,0.10,14258.33,12832.50,3503.51,Equipment Product 0107,Sports,Equipment,9328.99,14258.33,Premium
1,ORD000001,CUST00559,2026-05-28,Website,UPI,Completed,2026,5,2026-05,Customer 00559,...,0.05,4107.99,3902.59,453.62,Personal Care Product 0001,Beauty,Personal Care,3448.97,4107.99,Premium
2,ORD000001,CUST00559,2026-05-28,Website,UPI,Completed,2026,5,2026-05,Customer 00559,...,0.10,48513.80,43662.42,9232.80,Sportswear Product 0220,Sports,Sportswear,17214.81,24256.90,Value
3,ORD000002,CUST01197,2025-11-01,Website,Net Banking,Completed,2025,11,2025-11,Customer 01197,...,0.15,8940.37,7599.31,2277.79,Smartphones Product 0079,Electronics,Smartphones,5321.52,8940.37,Standard
4,ORD000003,CUST00400,2026-01-21,Website,Debit Card,Returned,2026,1,2026-01,Customer 00400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8232 entries, 0 to 8231
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             8232 non-null   str           
 1   customer_id          8232 non-null   str           
 2   order_date           8232 non-null   datetime64[us]
 3   sales_channel        8232 non-null   str           
 4   payment_method       8232 non-null   str           
 5   order_status         8232 non-null   str           
 6   year                 8232 non-null   int32         
 7   month                8232 non-null   int32         
 8   year_month           8232 non-null   period[M]     
 9   customer_name        8231 non-null   str           
 10  signup_date          8231 non-null   datetime64[us]
 11  segment              8231 non-null   str           
 12  city                 8231 non-null   str           
 13  region               8231 non-null   str    

## 8. Revenue / Profit / Margin by Product

In [28]:
product_analysis = df.groupby(["product_id","product_name"]).agg(total_revenue=("net_sales","sum"),total_profit=("profit","sum")).reset_index()

In [29]:
product_analysis["profit_margin_pct"] = (product_analysis["total_profit"]/product_analysis["total_revenue"]*100).round(2)

In [30]:
product_analysis = product_analysis.sort_values("total_revenue",ascending=False)
product_analysis.head(10)

,product_id,product_name,total_revenue,total_profit,profit_margin_pct
219,PROD0220,Sportswear Product 0220,1637340.79,363444.85,22.20
48,PROD0049,Accessories Product 0049,1559926.10,384841.31,24.67
165,PROD0166,Stationery Product 0166,1544414.30,475183.12,30.77
8,PROD0009,Decor Product 0009,1537549.51,506790.41,32.96
193,PROD0194,Storage Product 0194,1500886.98,493780.88,32.90
166,PROD0167,Appliances Product 0167,1488037.46,521382.32,35.04
69,PROD0070,Haircare Product 0070,1471829.77,488411.29,33.18
99,PROD0100,Accessories Product 0100,1452528.26,203677.11,14.02
127,PROD0128,Smartphones Product 0128,1435021.44,271476.12,18.92
70,PROD0071,Laptops Product 0071,1394716.99,489210.93,35.08


## 9. Monthly Revenue and MoM Growth

In [31]:
monthly_revenue = df.groupby("year_month").agg(total_revenue=("net_sales","sum")).reset_index()

In [32]:
monthly_revenue["previous_month_revenue"] = (monthly_revenue["total_revenue"].shift(1))

In [33]:
monthly_revenue["mom_growth_pct"] = (
    (monthly_revenue["total_revenue"] - monthly_revenue["previous_month_revenue"]) 
    / monthly_revenue["previous_month_revenue"] * 100).round(2)

In [34]:
monthly_revenue

,year_month,total_revenue,previous_month_revenue,mom_growth_pct
0,2024-01,4738502.21,NaN,NaN
1,2024-02,4484831.85,4738502.21,-5.35
2,2024-03,4089066.82,4484831.85,-8.82
3,2024-04,3741713.54,4089066.82,-8.49
4,2024-05,4534326.02,3741713.54,21.18
5,2024-06,3692885.22,4534326.02,-18.56
6,2024-07,4574839.56,3692885.22,23.88
7,2024-08,4365624.27,4574839.56,-4.57
8,2024-09,4374415.81,4365624.27,0.20
9,2024-10,5577687.02,4374415.81,27.51


## 10. Repeat Customer Analysis

In [35]:
customer_orders = (orders.groupby("customer_id").agg(order_count=("order_id","nunique")).reset_index())

In [36]:
repeat_customers = customer_orders[customer_orders["order_count"]>1]
repeat_customers.head()

,customer_id,order_count
0,CUST00001,4
1,CUST00002,5
2,CUST00003,4
4,CUST00005,2
5,CUST00006,3


In [37]:
repeat_customer_count = repeat_customers.shape[0]
print(repeat_customer_count)

1111


In [38]:
total_orders = customer_orders.shape[0]
repeat_customer_pct = (repeat_customer_count / total_orders * 100)
print(round(repeat_customer_pct,2))

93.99


## 11. RFM-style Recency / Frequency / Monetary features

In [39]:
frequency_date = orders["order_date"].max() + pd.Timedelta(days=1)

In [40]:
customer_latest = (
    df.groupby("customer_id").agg(last_order_date=("order_date","max")).reset_index()
)

In [41]:
customer_latest["recency"] = (frequency_date - customer_latest["last_order_date"]).dt.days

In [42]:
frequency = df.groupby("customer_id")["order_id"].nunique().reset_index(name="frequency")
customer_latest = customer_latest.merge(frequency,on="customer_id")

In [43]:
monetary = (df.groupby("customer_id")["net_sales"].sum().reset_index(name="monetary"))

In [44]:
customer_latest = customer_latest.merge(monetary,on="customer_id")
customer_latest.head()

,customer_id,last_order_date,recency,frequency,monetary
0,CUST00001,2026-04-30,125,4,130845.72
1,CUST00002,2026-04-24,131,5,268503.62
2,CUST00003,2025-05-14,476,4,121576.08
3,CUST00004,2026-01-23,222,1,1214.71
4,CUST00005,2025-01-08,602,2,23070.79


## 12. High revenue / low margin products

In [45]:
revenue_median = product_analysis["total_revenue"].median()
margin_median = product_analysis["profit_margin_pct"].median()

In [46]:
high_revenue_low_margin = product_analysis[
    (product_analysis["total_revenue"] > revenue_median) &
    (product_analysis["profit_margin_pct"] < margin_median)
]

In [47]:
high_revenue_low_margin = high_revenue_low_margin.sort_values(
    "total_revenue",
    ascending=False
)

## 13. Return reasons and category return rates

In [48]:
return_reasons = returns.groupby("return_reason").size().reset_index(name="return_count").sort_values("return_count",ascending=False)
return_reasons

,return_reason,return_count
1,Damaged Product,71
2,Late Delivery,65
5,Wrong Item,62
3,Quality Issue,55
4,Size/Fit Issue,52
0,Changed Mind,46


In [49]:
df_returns = df.merge(returns[["order_id","return_reason","refund_amount"]],on="order_id",how="left")

In [50]:
df_returns["is_returned"] = df_returns["return_reason"].notna()

In [51]:
category_returns = (
    df_returns.groupby("category")
    .agg(
        total_orders=("order_id", "nunique"),
        returned_orders=("is_returned", "sum")
    )
    .reset_index()
)

In [52]:
category_returns["return_rate_pct"] = (
    category_returns["returned_orders"]
    / category_returns["total_orders"]
    * 100
).round(2)
category_returns = category_returns.sort_values(
    "return_rate_pct",
    ascending=False
)
category_returns

,category,total_orders,returned_orders,return_rate_pct
0,Beauty,974,88,9.03
4,Office,974,81,8.32
2,Fashion,871,70,8.04
1,Electronics,1229,98,7.97
5,Sports,1141,86,7.54
3,Home & Kitchen,1052,73,6.94


## 14. Delivery-time outliers

In [53]:
Q1 = shipping["delivery_days"].quantile(0.25)
Q3 = shipping["delivery_days"].quantile(0.75)

In [54]:
IQR = Q3 - Q1
IQR

np.float64(4.0)

In [55]:
upper_limit = Q3 + 1.5 * IQR
delivery_outliers = shipping[
    shipping["delivery_days"] > upper_limit
]
delivery_outliers[
    ["order_id", "shipping_mode", "dispatch_date",
     "delivery_date", "delivery_days"]
].sort_values(
    "delivery_days",
    ascending=False
)
delivery_outliers

,order_id,shipping_mode,dispatch_date,delivery_date,shipping_cost,delivery_days


## 15. Create the analysis-ready dataset

In [56]:
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["year_month"] = df["order_date"].dt.to_period("M")

In [57]:
analysis_ready = df[
    [
        "order_id",
        "order_date",
        "year",
        "month",
        "year_month",
        "customer_id",
        "customer_name",
        "segment",
        "region",
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "quantity",
        "discount_pct",
        "net_sales",
        "profit"
    ]
].copy()

In [58]:
analysis_ready.shape

(8232, 17)

In [59]:
analysis_ready.info()

<class 'pandas.DataFrame'>
RangeIndex: 8232 entries, 0 to 8231
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       8232 non-null   str           
 1   order_date     8232 non-null   datetime64[us]
 2   year           8232 non-null   int32         
 3   month          8232 non-null   int32         
 4   year_month     8232 non-null   period[M]     
 5   customer_id    8232 non-null   str           
 6   customer_name  8231 non-null   str           
 7   segment        8231 non-null   str           
 8   region         8231 non-null   str           
 9   product_id     7000 non-null   str           
 10  product_name   7000 non-null   str           
 11  category       7000 non-null   str           
 12  subcategory    7000 non-null   str           
 13  quantity       7000 non-null   float64       
 14  discount_pct   7000 non-null   float64       
 15  net_sales      7000 non-null   f

In [60]:
analysis_ready.isna().sum()

order_id            0
order_date          0
year                0
month               0
year_month          0
customer_id         0
customer_name       1
segment             1
region              1
product_id       1232
product_name     1232
category         1232
subcategory      1232
quantity         1232
discount_pct     1232
net_sales        1232
profit           1232
dtype: int64

In [61]:
analysis_ready.head()

,order_id,order_date,year,month,year_month,customer_id,customer_name,segment,region,product_id,product_name,category,subcategory,quantity,discount_pct,net_sales,profit
0,ORD000001,2026-05-28,2026,5,2026-05,CUST00559,Customer 00559,Consumer,West,PROD0107,Equipment Product 0107,Sports,Equipment,1.0,0.10,12832.50,3503.51
1,ORD000001,2026-05-28,2026,5,2026-05,CUST00559,Customer 00559,Consumer,West,PROD0001,Personal Care Product 0001,Beauty,Personal Care,1.0,0.05,3902.59,453.62
2,ORD000001,2026-05-28,2026,5,2026-05,CUST00559,Customer 00559,Consumer,West,PROD0220,Sportswear Product 0220,Sports,Sportswear,2.0,0.10,43662.42,9232.80
3,ORD000002,2025-11-01,2025,11,2025-11,CUST01197,Customer 01197,Consumer,East,PROD0079,Smartphones Product 0079,Electronics,Smartphones,1.0,0.15,7599.31,2277.79
4,ORD000003,2026-01-21,2026,1,2026-01,CUST00400,Customer 00400,Consumer,East,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## EDA

In [63]:
print(analysis_ready["net_sales"].sum())

140529817.97


In [64]:
print(analysis_ready["profit"].sum())

34429654.03


In [65]:
print(analysis_ready["order_id"].nunique())

5000


In [66]:
print(analysis_ready["customer_id"].nunique())

1182


In [67]:
print(analysis_ready["quantity"].sum())

12442.0


## Monthly sales trend

In [68]:
monthly_sales = (analysis_ready.groupby("year_month").agg(
    revenue=("net_sales","sum"),
    profit=("profit","sum"),
    units=("quantity","sum"))).reset_index()
monthly_sales

,year_month,revenue,profit,units
0,2024-01,4738502.21,1191381.47,414.0
1,2024-02,4484831.85,1141478.40,401.0
2,2024-03,4089066.82,930787.64,368.0
3,2024-04,3741713.54,913158.05,311.0
4,2024-05,4534326.02,1113778.61,400.0
5,2024-06,3692885.22,888034.54,326.0
6,2024-07,4574839.56,1122929.22,428.0
7,2024-08,4365624.27,1008503.49,383.0
8,2024-09,4374415.81,1062628.01,378.0
9,2024-10,5577687.02,1271926.55,481.0


In [69]:
monthly_sales["profit_margin_pct"] = (
    monthly_sales["profit"] / monthly_sales["revenue"] * 100
).round(2)
monthly_sales

,year_month,revenue,profit,units,profit_margin_pct
0,2024-01,4738502.21,1191381.47,414.0,25.14
1,2024-02,4484831.85,1141478.40,401.0,25.45
2,2024-03,4089066.82,930787.64,368.0,22.76
3,2024-04,3741713.54,913158.05,311.0,24.40
4,2024-05,4534326.02,1113778.61,400.0,24.56
5,2024-06,3692885.22,888034.54,326.0,24.05
6,2024-07,4574839.56,1122929.22,428.0,24.55
7,2024-08,4365624.27,1008503.49,383.0,23.10
8,2024-09,4374415.81,1062628.01,378.0,24.29
9,2024-10,5577687.02,1271926.55,481.0,22.80


## Category performance

In [70]:
category_analysis = (analysis_ready.groupby("category").agg(
    revenue=("net_sales","sum"),
    profit=("profit","sum"),
    units=("quantity","sum")
    )
)
category_analysis["profit_margin_pct"] = (
    category_analysis["profit"] /
    category_analysis["revenue"] * 100
).round(2)

category_analysis.sort_values(
    "revenue",
    ascending=False
)

,revenue,profit,units,profit_margin_pct
category,,,,
Electronics,29306799.05,6874132.41,2428.0,23.46
Home & Kitchen,24424369.36,6914750.46,2147.0,28.31
Sports,24302957.98,6213955.81,2387.0,25.57
Office,22838355.34,5459776.06,1896.0,23.91
Beauty,20245393.98,4893703.40,1897.0,24.17
Fashion,19411942.26,4073335.89,1687.0,20.98


In [71]:
# 1. Duplicate rows
print("Duplicate orders:", orders.duplicated().sum())
print("Duplicate order items:", order_items.duplicated().sum())

# 2. Invalid customer IDs
print("\nInvalid customer IDs:")
print(orders[~orders["customer_id"].isin(customers["customer_id"])]["customer_id"].unique())

# 3. Orders without order items
print("\nOrders without order items:")
print((~orders["order_id"].isin(order_items["order_id"])).sum())

# 4. Order items without orders
print("\nOrder items without orders:")
print((~order_items["order_id"].isin(orders["order_id"])).sum())

# 5. Order items with invalid products
print("\nOrder items with invalid products:")
print((~order_items["product_id"].isin(products["product_id"])).sum())

# 6. Invalid quantities
print("\nQuantity <= 0:")
print((order_items["quantity"] <= 0).sum())

# 7. Invalid discounts
print("\nDiscount outside 0-1:")
print(
    (
        (order_items["discount_pct"] < 0) |
        (order_items["discount_pct"] > 1)
    ).sum()
)

# 8. Invalid shipping dates
print("\nDelivery before dispatch:")
print(
    (shipping["delivery_date"] < shipping["dispatch_date"]).sum()
)

# 9. Missing values
print("\nMissing values:")
for name, data in tables.items():
    print(f"\n{name}")
    print(data.isna().sum()[data.isna().sum() > 0])

Duplicate orders: 0
Duplicate order items: 0

Invalid customer IDs:
<StringArray>
['CUST99999']
Length: 1, dtype: str

Orders without order items:
1232

Order items without orders:
0

Order items with invalid products:
0

Quantity <= 0:
1

Discount outside 0-1:
1

Delivery before dispatch:
0

Missing values:

customers
Series([], dtype: int64)

products
Series([], dtype: int64)

orders_raw
Series([], dtype: int64)

orders_items_raw
Series([], dtype: int64)

payments
Series([], dtype: int64)

shipping
Series([], dtype: int64)

returns
Series([], dtype: int64)


## data quality issue

In [72]:
order_items[order_items["quantity"] <= 0]

,order_item_id,order_id,product_id,quantity,discount_pct,gross_sales,net_sales,profit
8,ITEM0000009,ORD004058,PROD0184,0,0.2,11181.6,8945.28,1772.58


In [73]:
order_items[
    (order_items["discount_pct"] < 0) |
    (order_items["discount_pct"] > 1)
]

,order_item_id,order_id,product_id,quantity,discount_pct,gross_sales,net_sales,profit
14,ITEM0000015,ORD001019,PROD0196,1,1.2,5795.45,5215.9,1090.62


In [74]:
order_items_clean = order_items[
    (order_items["quantity"] > 0) &
    (order_items["discount_pct"].between(0, 1))
].copy()

In [75]:
print("Original rows:", len(order_items))
print("Clean rows:", len(order_items_clean))
print("Removed rows:", len(order_items) - len(order_items_clean))

Original rows: 7000
Clean rows: 6998
Removed rows: 2


In [76]:
print("Invalid quantity:",
      (order_items_clean["quantity"] <= 0).sum())

print("Invalid discount:",
      ((order_items_clean["discount_pct"] < 0) |
       (order_items_clean["discount_pct"] > 1)).sum())

Invalid quantity: 0
Invalid discount: 0


In [77]:
sales_orders = orders[
    orders["order_id"].isin(order_items_clean["order_id"])
].copy()

In [78]:
print("Original orders:", orders["order_id"].nunique())
print("Orders with valid items:", sales_orders["order_id"].nunique())
print("Orders without items:",
      (~orders["order_id"].isin(order_items_clean["order_id"])).sum())

Original orders: 5000
Orders with valid items: 3768
Orders without items: 1232


In [79]:
powerbi_data = sales_orders.merge(
    customers,
    on="customer_id",
    how="left"
)

powerbi_data = powerbi_data.merge(
    order_items_clean,
    on="order_id",
    how="inner"
)

powerbi_data = powerbi_data.merge(
    products,
    on="product_id",
    how="left"
)

In [80]:
powerbi_data["year"] = powerbi_data["order_date"].dt.year
powerbi_data["month"] = powerbi_data["order_date"].dt.month
powerbi_data["year_month"] = (
    powerbi_data["order_date"]
    .dt.to_period("M")
    .astype(str)
)

In [81]:
powerbi_data.shape

(6998, 28)

In [82]:
powerbi_data.isna().sum()

order_id               0
customer_id            0
order_date             0
sales_channel          0
payment_method         0
order_status           0
year                   0
month                  0
year_month             0
customer_name          1
signup_date            1
segment                1
city                   1
region                 1
acquisition_channel    1
order_item_id          0
product_id             0
quantity               0
discount_pct           0
gross_sales            0
net_sales              0
profit                 0
product_name           0
category               0
subcategory            0
unit_cost              0
list_price             0
product_tier           0
dtype: int64

In [83]:
print("Rows:", len(powerbi_data))

print("Invalid quantity:",
      (powerbi_data["quantity"] <= 0).sum())

print("Invalid discount:",
      ((powerbi_data["discount_pct"] < 0) |
       (powerbi_data["discount_pct"] > 1)).sum())

print("Missing product IDs:",
      powerbi_data["product_id"].isna().sum())

print("Missing sales:",
      powerbi_data["net_sales"].isna().sum())

print("Missing profit:",
      powerbi_data["profit"].isna().sum())

Rows: 6998
Invalid quantity: 0
Invalid discount: 0
Missing product IDs: 0
Missing sales: 0
Missing profit: 0


## data export for power bi

In [84]:
powerbi_data.to_csv(
    "Ecommerce_PowerBI_Data.csv",
    index=False
)

In [85]:
powerbi_data.shape

(6998, 28)